In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Ensure directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

n = 2000 # Enough rows to test the pipeline
dates = [datetime(2026, 1, 1) + timedelta(hours=x*12) for x in range(n)]

# Generate dummy features that match your schema
df = pd.DataFrame({
    'shipment_id': [f'SC-{1000+i}' for i in range(n)],
    'order_date': sorted(dates),
    'supplier_name': np.random.choice(['TechCorp', 'GlobalSupply', 'AeroParts'], n),
    'supplier_id': np.random.choice(['S1', 'S2', 'S3'], n),
    'planned_delivery_date': sorted([d + timedelta(days=14) for d in dates]),
    'destination_city': np.random.choice(['Los Angeles', 'New York', 'Frankfurt'], n),
    'transportation_mode': np.random.choice(['Sea', 'Air', 'Rail'], n),
    'order_value_usd': np.random.uniform(5000, 85000, n).round(2),
    'warehouse_id': np.random.choice(['WH_A', 'WH_B'], n),
    'port_congestion_level_e': np.random.choice(['Low', 'Medium', 'High'], n),
    'sea_x_congestion': np.random.choice(['Low', 'High'], n),
    'sourcing_fragility': np.random.choice(['Low', 'High'], n),
    'customs_clearance_hours': np.random.choice(['Low', 'High'], n),
    'ext_risk': np.random.choice(['Low', 'High'], n),
    'supplier_id_hist': np.random.uniform(0, 1, n),
    
    # Target and Drop Columns
    'y': np.random.choice([0, 1], n, p=[0.56, 0.44]),
    'delay_days': np.random.randint(0, 15, n),
    'disruption_type': np.random.choice(['None', 'Weather', 'Customs'], n),
    'risk_score': np.random.uniform(0, 100, n),
    'actual_delivery_date': sorted([d + timedelta(days=16) for d in dates]),
    'notes': 'test data'
})

df.to_parquet('../data/processed/base_table.parquet', index=False)
print("✅ Ghost data generated at data/processed/base_table.parquet")

✅ Ghost data generated at data/processed/base_table.parquet


In [4]:
import shap
import pandas as pd
import numpy as np

print("🧠 Initializing SHAP TreeExplainer...")
# HistGradientBoosting doesn't always play perfectly with TreeExplainer in older sklearn versions,
# but we can use the universal explainer or TreeExplainer depending on the setup. 
# For HGB, we often use the exact Explainer.
explainer = shap.Explainer(model.predict, X_test)

# 1. Sample to save time (SHAP on massive datasets can take hours)
sample_size = min(2000, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=42)

print(f"Calculating SHAP values for {sample_size} rows. This might take a minute...")
shap_values = explainer(X_test_sample)

# 2. Extract the Top 3 Drivers per row
# Get the absolute SHAP values
shap_abs = np.abs(shap_values.values)

# Find the indices of the top 3 features for each prediction
top3_idx = np.argsort(-shap_abs, axis=1)[:, :3]

# Map those indices back to the actual feature column names
feature_names = X_test.columns
driver_df = pd.DataFrame({
    'driver_1': [feature_names[i] for i in top3_idx[:, 0]],
    'driver_2': [feature_names[i] for i in top3_idx[:, 1]],
    'driver_3': [feature_names[i] for i in top3_idx[:, 2]],
}, index=X_test_sample.index)

# 3. Merge real drivers back into your prediction dataframe
# (Assuming 'pred' is still in memory from the previous cell)
print("Mapping real drivers to predictions...")
pred_updated = pred.copy()

# Default to a global top driver just in case a row wasn't sampled
global_top_driver = feature_names[np.argmax(shap_abs.mean(axis=0))]
pred_updated['driver_1'] = global_top_driver 

# Overwrite with the specific per-row SHAP drivers we just calculated
pred_updated.loc[X_test_sample.index, 'driver_1'] = driver_df['driver_1']

# 4. Re-run the Actionable Rules Engine using REAL data
RULES = {
    ('port_congestion_level_e', 'High'): 'Pre-position stock; consider air freight',
    ('sea_x_congestion', 'High'): 'Switch transport mode to Air or Rail',
    ('sourcing_fragility', 'High'): 'Dual-source this lane; qualify a backup supplier',
    ('ext_risk', 'High'): 'Increase safety stock buffer; monitor weather corridor',
    ('supplier_id_hist', 'High'): 'Escalate to procurement; request delivery confirmation',
    ('customs_clearance_hours', 'High'): 'Pre-clear documentation; engage customs broker',
}
DEFAULT = {
    'High':   'Escalate to procurement and confirm next two deliveries',
    'Medium': 'Increase check-in frequency to twice weekly',
    'Low':    'Standard monitoring — no action required',
}

pred_updated['recommended_action'] = [
    RULES.get((d1, b), DEFAULT[b]) 
    for d1, b in zip(pred_updated['driver_1'], pred_updated['risk_band'])
]

# 5. Overwrite the outputs
pred_updated.to_csv('../outputs/predictions.csv', index=False)
print("✅ SHAP Integration Complete! outputs/predictions.csv updated with real drivers.")

Background dataset has 400 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=400 when initializing the masker.


🧠 Initializing SHAP TreeExplainer...
Calculating SHAP values for 400 rows. This might take a minute...


PermutationExplainer explainer: 401it [01:17,  4.91it/s]                                                                             

Mapping real drivers to predictions...
✅ SHAP Integration Complete! outputs/predictions.csv updated with real drivers.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import time

print("Starting the 5-Model Bakeoff...")

# 1. Define the 5 Models
models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "HistGradientBoosting (Champion)": HistGradientBoostingClassifier(random_state=42)
}

results = []

# 2. Train and Evaluate Each Model
for name, m in models.items():
    start_time = time.time()
    
    # Train
    m.fit(X_train, y_train)
    
    # Predict
    y_prob = m.predict_proba(X_test)[:, 1]
    
    # Evaluate
    roc = roc_auc_score(y_test, y_prob)
    pr = average_precision_score(y_test, y_prob)
    train_time = time.time() - start_time
    
    results.append({
        "Algorithm": name,
        "ROC-AUC": round(roc, 3),
        "PR-AUC": round(pr, 3),
        "Train Time (s)": round(train_time, 2)
    })

# 3. Generate the Leaderboard
leaderboard = pd.DataFrame(results).sort_values(by="PR-AUC", ascending=False).reset_index(drop=True)
print("\n🏆 MODEL BAKEOFF RESULTS 🏆")
display(leaderboard)

Starting the 5-Model Bakeoff...

🏆 MODEL BAKEOFF RESULTS 🏆


,Algorithm,ROC-AUC,PR-AUC,Train Time (s)
0,Decision Tree,0.551,0.472,0.01
1,Logistic Regression (Baseline),0.514,0.452,0.10
2,AdaBoost,0.512,0.451,0.08
3,Random Forest,0.479,0.448,0.13
4,HistGradientBoosting (Champion),0.474,0.426,0.14


In [5]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingClassifier
import time

print("⚙️ Starting 5-Minute Hyperparameter Tuning...")
start_time = time.time()

# 1. Define the bounded search grid for HistGradientBoosting
param_grid = {
    'max_iter': [100, 200, 300],          # Number of trees
    'learning_rate': [0.03, 0.05, 0.10],  # How fast the model learns
    'max_depth': [4, 6, 8],               # Complexity of each tree
    'min_samples_leaf': [10, 20, 50],     # Prevents overfitting on small leaves
    'l2_regularization': [0, 0.1, 1.0]    # Penalizes extreme weights
}

# 2. Use TimeSeriesSplit (Crucial for Forecasting)
# Standard Cross-Validation leaks future data into the past. 
# TimeSeriesSplit ensures the model always trains on the past and predicts the future.
tscv = TimeSeriesSplit(n_splits=3)

# 3. Setup the Randomized Search
# n_iter=15 means it will only try 15 random combinations instead of all 243.
search = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=15, 
    scoring='average_precision', # PR-AUC is better than ROC-AUC for imbalanced disruption data
    cv=tscv,
    n_jobs=-1, # Use all CPU cores
    random_state=42,
    verbose=1
)

# 4. Run the Search
search.fit(X_train, y_train)

# 5. Evaluate the Results
elapsed_time = (time.time() - start_time) / 60
print(f"\n✅ Tuning Complete in {elapsed_time:.1f} minutes!")
print(f"Best Parameters Found:\n{json.dumps(search.best_params_, indent=2)}")

# 6. Save the Champion Model
best_model = search.best_estimator_
joblib.dump(best_model, '../outputs/model_tuned.pkl')

print("🏆 The tuned model has been saved as 'model_tuned.pkl'.")
print("Update app.py to load this file for maximum accuracy!")

⚙️ Starting 5-Minute Hyperparameter Tuning...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

✅ Tuning Complete in 0.1 minutes!
Best Parameters Found:
{
  "min_samples_leaf": 10,
  "max_iter": 300,
  "max_depth": 6,
  "learning_rate": 0.1,
  "l2_regularization": 0.1
}
🏆 The tuned model has been saved as 'model_tuned.pkl'.
Update app.py to load this file for maximum accuracy!
